# AI-powered Resume Screening Assistant using LangChain and RAG 

## Project Scope and Evidence Policy

This project implements an AI-powered Resume Screening Assistant using LangChain and Retrieval-Augmented Generation (RAG).
The application compares Job Description requirements with evidence contained in uploaded resumes and generates structured candidate evaluations including match scores, matching and missing requirements, candidate summaries, strengths, weaknesses, and hiring recommendations.
The system follows a conservative evidence-grounded assessment policy:
- Candidate facts must come exclusively from the uploaded resume.
- A requirement is supported only when the resume provides direct affirmative evidence for it.
- Wording does not need to match the JD exactly, but the evidence must clearly describe the same capability.
- Related concepts are not automatically treated as equivalent.
- Qualifiers such as experience duration, advanced proficiency, work context, tools, education, and certifications must themselves be demonstrated.
- Elevated requirements such as “advanced”, “expert”, “extensive”, or “deep” are treated conservatively and are supported only when the resume clearly demonstrates that elevated level; ambiguous evidence is treated as not demonstrated.
- Missing or ambiguous evidence is treated as not demonstrated, rather than inferred.

**Scope disclaimer: This project is not intended to be a general-purpose semantic resume-understanding engine that infers unstated or transferable competencies. Its objective is to demonstrate a transparent, evidence-grounded LangChain + RAG resume-screening workflow.**

-----

### Step 1 — Import libraries, load API key & initialize LLM

In [1]:
# Step 1 — Import libraries, load API key & initialize LLM

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import Literal
from dotenv import load_dotenv
import os


load_dotenv()

api_key_loaded = bool(
    os.getenv("OPENAI_API_KEY")
)

assert api_key_loaded, (
    "OPENAI_API_KEY not found. "
    "Add it to .env before continuing."
)


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


print("OPENAI_API_KEY detected securely.")
print("LLM initialized: gpt-4o-mini")
print("Temperature: 0")

/var/folders/46/tx0_2b790hx5jt5ww66hfhc00000gn/T/ipykernel_5928/3411760483.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


OPENAI_API_KEY detected securely.
LLM initialized: gpt-4o-mini
Temperature: 0


### Step 2 — PDF Resume loading & text extraction
**Objective**: Load each resume PDF with LangChain's `PyPDFLoader` and confirm that text was extracted from every file.

In [2]:

resume_files = [
    "data/Resume_A_Data_Scientist.pdf",
    "data/Resume_B_Data_Scientist.pdf",
    "data/Resume_C_ML_Engineer.pdf"
]

resume_documents = {}

for file_path in resume_files:
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    resume_documents[file_path] = documents

    extracted_text = "\n".join(doc.page_content for doc in documents).strip()

    print(f"File: {file_path}")
    print(f"Pages loaded: {len(documents)}")
    print(f"Characters extracted: {len(extracted_text)}")
    print(f"Text preview: {extracted_text[:200]}")
    print("-" * 80)

File: data/Resume_A_Data_Scientist.pdf
Pages loaded: 1
Characters extracted: 4064
Text preview: AARAV MEHTA
Data Scientist
Bengaluru, India | +91 90000 12345 | aarav.mehta.ds@example.com | linkedin.com/in/aarav-mehta-ds
PROFESSIONAL SUMMARY
Data Scientist with 6 years of experience applying stat
--------------------------------------------------------------------------------
File: data/Resume_B_Data_Scientist.pdf
Pages loaded: 1
Characters extracted: 5060
Text preview: RHEA SHARMA
SENIOR DATA SCIENTIST
Bengaluru, India | +91 00000 00000 | rhea.sharma@example.com
PROFESSIONAL SUMMARY
Data Scientist with 9+ years of experience applying statistics, mathematics, data mi
--------------------------------------------------------------------------------
File: data/Resume_C_ML_Engineer.pdf
Pages loaded: 1
Characters extracted: 3261
Text preview: ARJUN MEHTA
 Machine Learning Engineer | Bengaluru, India | arjun.mehta@example.com | +91 98765 43210 | linkedin.com/in/arjunmehta-ml
PROFESSIONAL SUMM

### Step 3 — Candidate identification & resume isolation

**Objective**: Associate every loaded resume with a unique candidate so that later retrieval for one candidate can never accidentally use another candidate’s resume. This is the foundation for creating independent candidate vector stores later, to prevent cross-candidate referencing issue. 

In [3]:
candidate_map = {
    "data/Resume_A_Data_Scientist.pdf": {
        "candidate_id": "A",
        "candidate_name": "Aarav Mehta"
    },
    "data/Resume_B_Data_Scientist.pdf": {
        "candidate_id": "B",
        "candidate_name": "Rhea Sharma"
    },
    "data/Resume_C_ML_Engineer.pdf": {
        "candidate_id": "C",
        "candidate_name": "Arjun Mehta"
    }
}

candidate_documents = {}

for file_path, documents in resume_documents.items():
    candidate_info = candidate_map[file_path]

    for doc in documents:
        doc.metadata["candidate_id"] = candidate_info["candidate_id"]
        doc.metadata["candidate_name"] = candidate_info["candidate_name"]
        doc.metadata["source_file"] = file_path

    candidate_documents[candidate_info["candidate_id"]] = documents

    print(
        f"Candidate {candidate_info['candidate_id']}: "
        f"{candidate_info['candidate_name']} | "
        f"Documents: {len(documents)} | "
        f"Source: {file_path}"
    )

Candidate A: Aarav Mehta | Documents: 1 | Source: data/Resume_A_Data_Scientist.pdf
Candidate B: Rhea Sharma | Documents: 1 | Source: data/Resume_B_Data_Scientist.pdf
Candidate C: Arjun Mehta | Documents: 1 | Source: data/Resume_C_ML_Engineer.pdf


### Step 4 — Split resume text into chunks

**Objective:** Split each candidate’s resume into smaller overlapping text chunks while preserving the candidate metadata attached in Step 3.
This is necessary because embeddings and retrieval work better on smaller relevant sections of text rather than treating an entire resume as one large document.

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

candidate_chunks = {}

for candidate_id, documents in candidate_documents.items():
    chunks = text_splitter.split_documents(documents)
    candidate_chunks[candidate_id] = chunks

    print(
        f"Candidate {candidate_id}: "
        f"{chunks[0].metadata['candidate_name']} | "
        f"Chunks created: {len(chunks)}"
    )

    print(
        f"Metadata check: "
        f"{chunks[0].metadata['candidate_id']} | "
        f"{chunks[0].metadata['candidate_name']} | "
        f"{chunks[0].metadata['source_file']}"
    )

    print("-" * 80)

Candidate A: Aarav Mehta | Chunks created: 5
Metadata check: A | Aarav Mehta | data/Resume_A_Data_Scientist.pdf
--------------------------------------------------------------------------------
Candidate B: Rhea Sharma | Chunks created: 7
Metadata check: B | Rhea Sharma | data/Resume_B_Data_Scientist.pdf
--------------------------------------------------------------------------------
Candidate C: Arjun Mehta | Chunks created: 4
Metadata check: C | Arjun Mehta | data/Resume_C_ML_Engineer.pdf
--------------------------------------------------------------------------------


### Step 5 — Create OpenAI embeddings
**Objective**: Initialize the OpenAI embedding model using text-embedding-3-small and verify that it can successfully convert actual resume text into a numerical vector.

In [5]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

test_embedding = embeddings.embed_query(
    candidate_chunks["A"][0].page_content
)

print("Embedding model: text-embedding-3-small")
print(f"Test chunk candidate: {candidate_chunks['A'][0].metadata['candidate_name']}")
print(f"Embedding created: {len(test_embedding) > 0}")
print(f"Embedding dimensions: {len(test_embedding)}")

Embedding model: text-embedding-3-small
Test chunk candidate: Aarav Mehta
Embedding created: True
Embedding dimensions: 1536


### Step 6 — Create FAISS vector stores
**Objective**: Create a separate FAISS vector store for each candidate using that candidate’s own resume chunks and the validated OpenAI embedding model.
This keeps the candidates isolated and prevents Resume A evidence from being retrieved for Resume B or C.

In [6]:
candidate_vectorstores = {}

for candidate_id, chunks in candidate_chunks.items():
    vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    candidate_vectorstores[candidate_id] = vectorstore

    print(
        f"Candidate {candidate_id}: "
        f"{chunks[0].metadata['candidate_name']} | "
        f"Chunks indexed: {len(chunks)} | "
        f"Vector store created: {vectorstore is not None}"
    )

Candidate A: Aarav Mehta | Chunks indexed: 5 | Vector store created: True
Candidate B: Rhea Sharma | Chunks indexed: 7 | Vector store created: True
Candidate C: Arjun Mehta | Chunks indexed: 4 | Vector store created: True


### Step 7 — Create retrievers & validate candidate isolation
**Objective** - Convert each candidate’s FAISS vector store into a retriever and verify that a JD-related query returns chunks only from the correct candidate.

Each resume is a single page that splits into four to seven chunks, so the retriever returns all of a candidate’s chunks, ranked by relevance to the query. This avoids missing evidence on short documents; for longer documents `k` would be reduced.

In [7]:
candidate_retrievers = {}

retrieval_query = """
data science experience, statistical analysis, predictive modeling,
Python, SQL, Tableau, Excel, PowerPoint, model deployment,
data mining, experimentation, stakeholder communication,
project management and production collaboration
"""

for candidate_id, vectorstore in candidate_vectorstores.items():

    k = len(candidate_chunks[candidate_id])
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )

    candidate_retrievers[candidate_id] = retriever
    retrieved_docs = retriever.invoke(retrieval_query)

    print(
        f"\nCandidate {candidate_id}: "
        f"{candidate_chunks[candidate_id][0].metadata['candidate_name']}"
    )
    print(
        f"Retrieved chunks: {len(retrieved_docs)} / "
        f"{len(candidate_chunks[candidate_id])}"
    )

    candidate_ids = {
        doc.metadata["candidate_id"]
        for doc in retrieved_docs
    }

    print(f"Candidate isolation check: {candidate_ids}")
    print("-" * 80)


Candidate A: Aarav Mehta
Retrieved chunks: 5 / 5
Candidate isolation check: {'A'}
--------------------------------------------------------------------------------

Candidate B: Rhea Sharma
Retrieved chunks: 7 / 7
Candidate isolation check: {'B'}
--------------------------------------------------------------------------------

Candidate C: Arjun Mehta
Retrieved chunks: 4 / 4
Candidate isolation check: {'C'}
--------------------------------------------------------------------------------


### Step 8 — Define structured evaluation schema & output parser
**Objective** - Define the exact structure that every final candidate evaluation must follow.

The final evaluation is assembled in Python in Step 17. The match score, matching and missing requirements, strengths, weaknesses and hiring recommendation are derived from the grounded requirement assessments; the LLM contributes only the candidate summary and the recommendation justification (Step 16).

A LangChain output parser is also created for this schema. The pipeline’s LLM calls use the dedicated parsers defined in Steps 9, 11 and 16.

Our required evaluation fields are:
- candidate name
- match score
- matching skills
- missing skills
- candidate summary
- strengths
- weaknesses
- hiring recommendation
- recommendation justification

In [8]:
class CandidateEvaluation(BaseModel):
    candidate_name: str = Field(
        description="Name of the candidate"
    )
    match_score: float = Field(
        ge=0,
        le=100,
        description="Candidate's JD requirement coverage score from 0 to 100"
    )
    matching_skills: list[str] = Field(
        description="Skills or requirements demonstrated in the resume"
    )
    missing_skills: list[str] = Field(
        description="JD skills or requirements not demonstrated in the resume"
    )
    candidate_summary: str = Field(
        description="Brief evidence-based summary of the candidate"
    )
    strengths: list[str] = Field(
        description="Candidate strengths supported by resume evidence"
    )
    weaknesses: list[str] = Field(
        description="Candidate weaknesses or gaps relative to the JD"
    )
    hiring_recommendation: str = Field(
        description="Hiring recommendation for the candidate"
    )
    recommendation_justification: str = Field(
        description="Evidence-based justification for the recommendation"
    )


evaluation_parser = PydanticOutputParser(
    pydantic_object=CandidateEvaluation
)

print("Structured output parser created successfully.")
print("\nRequired fields:")
for field_name in CandidateEvaluation.model_fields:
    print(f"- {field_name}")

Structured output parser created successfully.

Required fields:
- candidate_name
- match_score
- matching_skills
- missing_skills
- candidate_summary
- strengths
- weaknesses
- hiring_recommendation
- recommendation_justification


### Step 9 — Define JD requirement extraction schema
**Objective** -  Define a simple, structured representation of scorable JD requirements so that the next step can extract them consistently from any entered Job Description.
This step does not:
- evaluate resumes,
- retrieve resume evidence,
- calculate a score,
- or generate candidate conclusions.
It only defines what a JD requirement looks like.

In [9]:
class JDRequirement(BaseModel):
    requirement: str = Field(
        description="One atomic candidate requirement stated in the Job Description"
    )

    requirement_type: Literal[
        "required",
        "preferred"
    ] = Field(
        description="Whether the JD presents the requirement as required or preferred"
    )

    category: Literal[
        "skill",
        "tool",
        "experience",
        "education",
        "certification"
    ] = Field(
        description="The type of candidate requirement"
    )


class JDRequirements(BaseModel):
    requirements: list[JDRequirement] = Field(
        description="Atomic required and preferred candidate requirements extracted from the Job Description"
    )


jd_parser = PydanticOutputParser(
    pydantic_object=JDRequirements
)


print("JD requirement schema created successfully.")

print("\nJDRequirement fields:")
for field_name in JDRequirement.model_fields:
    print(f"- {field_name}")

print("\nJDRequirements fields:")
for field_name in JDRequirements.model_fields:
    print(f"- {field_name}")

JD requirement schema created successfully.

JDRequirement fields:
- requirement
- requirement_type
- category

JDRequirements fields:
- requirements


### Step 10 — Extract and normalize JD requirements
**Objective:** Use gpt-4o-mini + LangChain structured output parsing to convert the entered JD into the JDRequirements structure defined in Step 9.

The extraction runs in two passes: the first pass extracts the requirements, and a second audit pass checks the draft against the original JD for coverage and atomicity.

The extraction must:
- use only requirements actually stated in the JD,
- split compound requirements into atomic items,
- classify each as required or preferred,
- assign the Step 9 category,
- remove genuine duplicates,
- exclude general job responsibilities/objectives from this scoring list,
- not invent requirements.

In [10]:
#Pre-requisite: Job Description text is available in the variable `job_description` as a string.

job_description = """
At Company X, we rely on insightful data to power our systems and solutions. We’re seeking an experienced data scientist to deliver insights on a daily basis. The ideal candidate will have mathematical and statistical expertise, along with natural curiosity and a creative mind. While mining, interpreting, and cleaning our data, this person will be relied on to ask questions, connect the dots, and uncover hidden opportunities for realizing the data’s full potential. As part of a team of specialists, the data scientist will “slice and dice” data using various methods and create new visions for the future.
Objectives of this role
* Collaborate with product design and engineering teams to develop an understanding of needs
* Research and devise innovative statistical models for data analysis
* Communicate findings to all stakeholders
* Enable smarter business processes by using analytics for meaningful insights
* Keep current with technical and industry developments
Responsibilities
* Serve as lead data strategist to identify and integrate new datasets that can be leveraged through our product capabilities, and work closely with the engineering team in the development of data products
* Execute analytical experiments to help solve problems across various domains and industries
* Identify relevant data sources and sets to mine for client business needs, and collect large structured and unstructured datasets and variables
* Devise and utilize algorithms and models to mine big-data stores; perform data and error analysis to improve models; clean and validate data for uniformity and accuracy
* Analyze data for trends and patterns, and interpret data with clear objectives in mind
* Implement analytical models in production by collaborating with software developers and machine-learning engineers
Required skills and qualifications
* Seven or more years of experience in data science
* Proficiency with data mining, mathematics, and statistical analysis
* Advanced experience in pattern recognition and predictive modeling
* Experience with Excel, PowerPoint, Tableau, SQL, and programming languages (ex: Java/Python, SAS)
* Ability to work effectively in a dynamic, research-oriented group that has several concurrent projects
Preferred skills and qualifications
* Bachelor’s degree (or equivalent) in statistics, applied mathematics, or related discipline
* Two or more years of project management experience
* Professional certification
"""

In [11]:
# Step 10 — Extract, normalize and audit atomic JD requirements

jd_extraction_prompt = ChatPromptTemplate.from_template("""
You are extracting candidate requirements from a Job Description.

Extract ONLY requirements that can be used to evaluate whether a candidate
meets the Job Description.

Do NOT convert general responsibilities into candidate requirements.

ATOMICITY RULE

Each requirement must be independently scorable.

Ask:

"Could one meaningful part of this requirement be supported by a resume
while another meaningful part is absent?"

If YES, split it into separate requirements.

IMPORTANT:
When splitting a requirement, NEVER DROP any meaningful part of the
original requirement.

Example:

"Ability to work effectively in a dynamic, research-oriented group that
has several concurrent projects"

must produce BOTH:

- Ability to work effectively in a dynamic, research-oriented group
- Ability to work effectively on several concurrent projects

Another example:

"Proficiency with data mining, mathematics, and statistical analysis"

must produce:

- Proficiency with data mining
- Proficiency with mathematics
- Proficiency with statistical analysis

RULES

1. Preserve whether each requirement is REQUIRED or PREFERRED.

2. Split independent mandatory skills or capabilities.

3. Do NOT split alternatives or examples.

   Example:

   "Experience with programming languages (ex: Java/Python, SAS)"

   remains ONE requirement because the languages are examples or
   alternatives.

4. Preserve meaningful:
   - quantities
   - years
   - thresholds
   - qualification levels
   - degree requirements
   - certifications
   - words such as "advanced" or "research-oriented"

5. Do not weaken or strengthen the JD.

6. Do not invent requirements.

7. Remove duplicate requirements.

8. Assign the most appropriate category:
   - skill
   - tool
   - experience
   - education
   - certification

JOB DESCRIPTION:
{job_description}

{format_instructions}
""")


jd_extraction_chain = (
    jd_extraction_prompt
    | llm
    | jd_parser
)


draft_jd_requirements = jd_extraction_chain.invoke({
    "job_description": job_description,
    "format_instructions": jd_parser.get_format_instructions()
})


# PASS 2 — Audit atomicity and coverage

jd_audit_prompt = ChatPromptTemplate.from_template("""
You are auditing an extracted set of Job Description requirements.

Compare the ORIGINAL JOB DESCRIPTION with the DRAFT REQUIREMENTS.

Your job is to return the FINAL corrected requirement list.

AUDIT RULES

1. Every candidate qualification stated in the original JD must be represented.

2. No meaningful requirement may disappear when a compound requirement
   is split.

3. Each final requirement must be independently scorable.

Use this atomicity test:

"Could one meaningful part be supported by a resume while another
meaningful part is absent?"

If YES, split the requirement.

Example:

"Ability to work effectively in a dynamic, research-oriented group that
has several concurrent projects"

must result in BOTH:

- Ability to work effectively in a dynamic, research-oriented group
- Ability to work effectively on several concurrent projects

4. Split independent mandatory skills.

Example:

"Proficiency with data mining, mathematics, and statistical analysis"

must result in three independently scorable requirements.

5. Do NOT split alternatives or examples.

Example:

"Experience with programming languages (ex: Java/Python, SAS)"

remains one requirement.

6. Preserve all meaningful quantities, levels and qualifications including:
   - years of experience
   - "advanced"
   - degree requirements
   - certifications
   - "research-oriented"
   - other explicit thresholds

7. Preserve REQUIRED versus PREFERRED correctly.

8. Do not invent requirements that are not present in the original JD.

9. Do not convert general job responsibilities into candidate qualifications.

10. Remove duplicates.

ORIGINAL JOB DESCRIPTION:
{job_description}

DRAFT REQUIREMENTS:
{draft_requirements}

{format_instructions}
""")


jd_audit_chain = (
    jd_audit_prompt
    | llm
    | jd_parser
)


jd_requirements = jd_audit_chain.invoke({
    "job_description": job_description,
    "draft_requirements": draft_jd_requirements.model_dump_json(indent=2),
    "format_instructions": jd_parser.get_format_instructions()
})

# VALIDATION

assert jd_requirements.requirements, (
    "No JD requirements were extracted."
)

requirement_keys = [
    (
        item.requirement.strip().lower(),
        item.requirement_type,
        item.category
    )
    for item in jd_requirements.requirements
]

assert len(requirement_keys) == len(set(requirement_keys)), (
    "Duplicate JD requirements detected."
)


print("JD requirements extracted and audited successfully.\n")

for i, item in enumerate(
    jd_requirements.requirements,
    start=1
):
    print(
        f"{i:02d}. "
        f"{item.requirement_type.upper()} | "
        f"{item.category} | "
        f"{item.requirement}"
    )

print(
    f"\nTotal requirements: "
    f"{len(jd_requirements.requirements)}"
)

JD requirements extracted and audited successfully.

01. REQUIRED | experience | Seven or more years of experience in data science
02. REQUIRED | skill | Proficiency with data mining
03. REQUIRED | skill | Proficiency with mathematics
04. REQUIRED | skill | Proficiency with statistical analysis
05. REQUIRED | skill | Advanced experience in pattern recognition
06. REQUIRED | skill | Advanced experience in predictive modeling
07. REQUIRED | tool | Experience with Excel
08. REQUIRED | tool | Experience with PowerPoint
09. REQUIRED | tool | Experience with Tableau
10. REQUIRED | tool | Experience with SQL
11. REQUIRED | tool | Experience with programming languages (ex: Java/Python, SAS)
12. REQUIRED | skill | Ability to work effectively in a dynamic, research-oriented group
13. REQUIRED | skill | Ability to work effectively on several concurrent projects
14. PREFERRED | education | Bachelor’s degree (or equivalent) in statistics, applied mathematics, or related discipline
15. PREFERRED | e

### Step 11 — Define requirement evidence-assessment schema

**Objective** - Define a structured evidence-assessment record for one normalized JD requirement.

The record holds:
- the requirement metadata produced in Step 10 (requirement, type, category);
- the model’s findings: the core capability, the resume term it matched, whether that term is the same capability, only related, only an aspiration, or absent, and any qualifier with its evidence;
- supporting evidence quotes and a short reason;
- the final status: supported or not_supported.

The findings come before the status, so the model records what it found before any verdict. The status itself is recalculated in Python in Step 13.

In [12]:
# Step 11 — Define structured requirement evidence assessment schema

class RequirementEvidenceAssessment(BaseModel):

    requirement: str = Field(
        description="The JD requirement, copied exactly as supplied."
    )

    requirement_type: Literal["required", "preferred"] = Field(
        description="Copied exactly as supplied."
    )

    category: Literal[
        "skill",
        "tool",
        "experience",
        "education",
        "certification"
    ] = Field(
        description="Copied exactly as supplied."
    )

    core_capability: str = Field(
        description=(
            "The capability, tool, qualification or experience the "
            "requirement names, with level words, thresholds and "
            "subjective descriptors removed."
        )
    )

    resume_term: str = Field(
        description=(
            "Search the whole resume context first. If the resume uses "
            "the core capability's own name anywhere, resume_term must be "
            "that exact word or phrase. Otherwise use the closest phrase "
            "the resume contains. Use 'none' if the resume contains "
            "nothing relevant."
        )
    )

    capability_match: Literal[
        "same",
        "related_only",
        "aspiration_only",
        "absent"
    ] = Field(
        description=(
            "'same' if resume_term names the core capability itself, "
            "names one of the alternatives or examples listed in the "
            "requirement, or describes the candidate doing that same thing. "
            "Naming it as something the candidate applies, uses or is "
            "skilled in is enough. Do not require the resume to state a "
            "proficiency level or to elaborate. "
            "'related_only' if resume_term is a different skill, even one "
            "that normally relies on the core capability. "
            "'aspiration_only' if the resume only expresses interest or a "
            "wish to gain it. "
            "'absent' if resume_term is 'none'."
        )
    )

    qualifier: str = Field(
        description=(
            "The level word, numeric threshold, named field or concrete "
            "context the requirement adds to the core capability. Use "
            "'none' if there is none. General wording such as proficiency, "
            "experience or ability, and subjective descriptors such as "
            "dynamic or effectively, are not qualifiers."
        )
    )

    qualifier_kind: Literal[
        "none",
        "elevated_level",
        "numeric_threshold",
        "named_item",
        "concrete_context"
    ] = Field(
        description="The kind of qualifier identified above."
    )

    qualifier_evidence: str = Field(
        description=(
            "The exact resume quote that satisfies the qualifier, or "
            "'none'. For an elevated level, the quote itself must state "
            "that level for this capability or show elevated scope, "
            "complexity or responsibility in it. A quote that only shows "
            "the capability being used is not qualifier evidence."
        )
    )

    qualifier_met: bool = Field(
        description=(
            "True only if qualifier_evidence is not 'none' and satisfies "
            "the qualifier. False if qualifier_kind is 'none'."
        )
    )

    supporting_evidence: list[str] = Field(
        description=(
            "Exact quotes copied from the resume context that justify the "
            "findings above. Empty only when nothing relevant exists."
        )
    )

    assessment_reason: str = Field(
        description=(
            "One or two sentences explaining the findings above."
        )
    )

    status: Literal["supported", "not_supported"] = Field(
        description="Overall result. This is recalculated in Python."
    )


evidence_parser = PydanticOutputParser(
    pydantic_object=RequirementEvidenceAssessment
)

print(
    "Requirement evidence assessment schema created successfully."
)

Requirement evidence assessment schema created successfully.


### Step 12 — Create grounded evidence-assessment prompt

**Objective**
Create the prompt that will later evaluate one normalized JD requirement against retrieved context from one candidate’s resume only.
The prompt must:
- treat the JD requirement as evaluation criteria, never as evidence;
- use only retrieved resume context for candidate claims;
- first find the core capability, then check each qualifier (general wording, elevated level, numeric threshold, named item, concrete context) against its own rule;
- return the exact Step 11 structured format, with evidence quoted from the resume;
- avoid assuming skills, experience or qualifications not demonstrated by the resume.

The requirement metadata is attached, and the final supported / not_supported status is derived, in Python in Step 13.

This step does not retrieve documents, call the LLM, calculate scores, or evaluate a candidate yet.

In [13]:
# Step 12 — Create grounded evidence-assessment prompt

evidence_prompt = ChatPromptTemplate.from_template("""
You are assessing ONE Job Description requirement against the resume of ONE candidate.

The JD requirement is the evaluation criterion. It is never evidence about the candidate.
Use ONLY the resume context below. Do not use outside knowledge.

HOW TO DECIDE

Step A - Find the capability.

Identify the core capability, tool, qualification or experience the requirement names.

It counts as present only if the resume:
- names it directly, or
- describes the candidate actually doing the same thing.

Different wording is acceptable when it clearly describes the same capability.

It does NOT count when:
- it is only implied by a different skill;
- doing X would normally rely on Y, but Y itself is not demonstrated;
- the resume only expresses interest, intent, aspiration, or a wish to gain exposure to it;
- the evidence is ambiguous.


Step B - Check each qualifier using the matching rule.

1. General capability wording
   Examples:
   - "proficiency with"
   - "experience with"
   - "ability to"
   - "knowledge of"

   These are satisfied when Step A is satisfied.

   The resume does NOT need to repeat words such as:
   - proficiency
   - experience
   - ability
   - knowledge

   Direct and unambiguous evidence of actually using or demonstrating the same capability
   is sufficient.


2. Elevated level wording
   Examples:
   - "advanced"
   - "expert"
   - "extensive"
   - "deep"

   These are satisfied only when the resume:
   - explicitly states that elevated level for the same capability, OR
   - provides direct and unambiguous evidence of an elevated level through the
     scope, complexity, responsibility, or depth of the work described.

   Repeated mentions or ordinary use alone do NOT prove an elevated level.

   If the elevated level is ambiguous, return "not_supported".


3. Numeric thresholds
   Examples:
   - years of experience
   - counts
   - minimum quantities

   These are satisfied only when:
   - a number stated in the resume, OR
   - employment dates shown in the resume

   meet the threshold for that specific kind of experience.

   State the number or duration you found in assessment_reason.

   Do not transfer years from one type of experience to another.


4. Named items
   Examples:
   - tools
   - programming languages
   - degrees
   - fields of study
   - certifications

   These must be named in the resume.

   When the requirement lists alternatives or examples, one of them is enough.

   When the JD explicitly allows "related" or "equivalent" items, accept only a
   resume qualification whose wording is clearly and directly related to the
   named field.

   If the relationship is ambiguous, return "not_supported".


5. Concrete context
   Examples:
   - a type of team
   - work setting
   - domain
   - responsibility
   - project environment

   The concrete context must be described in the resume.

   Different wording is acceptable when it clearly describes the same context.


6. Subjective descriptors
   Examples:
   - "dynamic"
   - "fast-paced"
   - "effectively"

   These are not separate factual qualifiers unless the JD makes them objectively
   measurable.

   Judge the concrete parts of the requirement instead.


Step C - Decide.

Return "supported" ONLY if:
- Step A is satisfied, AND
- every applicable qualifier rule above is satisfied.

Otherwise return "not_supported".

"not_supported" means only that the resume does not demonstrate the complete
requirement.

Do NOT claim that the candidate definitely lacks the capability.


EVIDENCE RULES

- supporting_evidence must contain exact quotes copied character-for-character
  from the supplied resume context.
- Do NOT paraphrase.
- Do NOT merge separate passages.
- Do NOT rewrite, clean, or improve the wording.
- Each quote must be one continuous passage of at most 30 words.
- For "supported":
  provide 1 to 3 quotes that prove the requirement.
- For "not_supported":
  provide the closest relevant quote if one exists, such as:
  - a stated number below a threshold;
  - a weaker or related capability;
  - aspirational wording.
- Use an empty list only when the resume contains no relevant evidence.


ASSESSMENT REASON

assessment_reason must:
- be one or two concise sentences;
- explain which rule determined the result;
- refer only to the JD requirement and supplied resume evidence;
- not introduce outside facts or assumptions.


METADATA

Copy these values exactly as supplied:
- requirement
- requirement_type
- category


JD REQUIREMENT:
{requirement}

REQUIREMENT TYPE:
{requirement_type}

CATEGORY:
{category}

RESUME CONTEXT:
{resume_context}

{format_instructions}
""")

### Step 13 — Assess JD requirements against one candidate

**Assessed limitation with gpt-4o-mini**: After multiple trial and errors, I realised that `gpt-4o-mini` may occasionally misclassify requirements involving numeric thresholds or closely related experience wording, even when relevant evidence is present in the resume. These cases are exposed in the Step 18 audit trail and are conservatively counted as not demonstrated. For example "9+ years" against "seven or more years" will show as "not demonstrated" for candidate assessment, and the audit trail in step 18 will print the model's reason for it.

In [14]:
# Step 13 — Assess JD requirements against one candidate

evidence_chain = evidence_prompt | llm | evidence_parser


def assess_requirement(item, retriever):
    """
    Assess one JD requirement against one candidate's resume.

    The LLM extracts structured evidence findings.
    Python then derives the final supported / not_supported status
    deterministically from those findings.
    """

    # Retrieve evidence only from this candidate's resume
    retrieved_docs = retriever.invoke(item.requirement)

    resume_context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    # Let the LLM analyse capability, qualifier and evidence
    assessment = evidence_chain.invoke({
        "requirement": item.requirement,
        "requirement_type": item.requirement_type,
        "category": item.category,
        "resume_context": resume_context,
        "format_instructions": evidence_parser.get_format_instructions()
    })

    # Final status is derived from the structured findings, not trusted directly from the LLM.
    supported = (
        assessment.capability_match == "same"
        and (
            assessment.qualifier_kind == "none"
            or assessment.qualifier_met
        )
    )

    # Step 10 metadata remains authoritative. Python also overwrites the final status deterministically.
    return assessment.model_copy(update={
        "requirement": item.requirement,
        "requirement_type": item.requirement_type,
        "category": item.category,
        "status": "supported" if supported else "not_supported"
    })


def assess_candidate(candidate_id):
    """
    Assess all JD requirements independently against one candidate.
    """

    retriever = candidate_retrievers[candidate_id]

    return [
        assess_requirement(item, retriever)
        for item in jd_requirements.requirements
    ]


# Test Step 13 on Candidate A first
test_candidate_id = "A"

candidate_requirement_assessments = assess_candidate(
    test_candidate_id
)

candidate_name = (
    candidate_chunks[test_candidate_id][0]
    .metadata["candidate_name"]
)

print(
    f"Requirement assessments completed for "
    f"{candidate_name}.\n"
)


for i, a in enumerate(
    candidate_requirement_assessments,
    start=1
):
    print(
        f"{i:02d}. "
        f"{a.requirement_type.upper()} | "
        f"{a.status.upper()} | "
        f"{a.requirement}"
    )

    print(
        f"    Findings: resume_term='{a.resume_term}' | "
        f"match={a.capability_match} | "
        f"qualifier='{a.qualifier}' "
        f"({a.qualifier_kind}) | "
        f"met={a.qualifier_met}"
    )

    print(
        f"    Reason: "
        f"{a.assessment_reason}"
    )

    for evidence in a.supporting_evidence:
        print(
            f"    Evidence: "
            f"{evidence}"
        )

    print()


print(
    f"Total assessments: "
    f"{len(candidate_requirement_assessments)}"
)

Requirement assessments completed for Aarav Mehta.

01. REQUIRED | NOT_SUPPORTED | Seven or more years of experience in data science
    Findings: resume_term='6 years of experience' | match=related_only | qualifier='7' (numeric_threshold) | met=False
    Reason: The resume states 6 years of experience, which does not meet the 7 years required by the JD.
    Evidence: Data Scientist with 6 years of experience applying statistical analysis, predictive modeling, and machine learning to product, customer, and operational data.

02. REQUIRED | SUPPORTED | Proficiency with data mining
    Findings: resume_term='data mining' | match=same | qualifier='none' (none) | met=False
    Reason: The resume directly mentions 'data mining', satisfying the core capability requirement. There are no additional qualifiers present.
    Evidence: Skilled in Python, SQL, Tableau, Excel, experiment design, data mining, feature engineering, model evaluation, and communicating analytical findings to technical an

### Step 14 — Calculate Deterministic Match Score
**Objective:** Calculate a reproducible 0–100 match score in Python from the grounded Step 13 assessments. The score is calculated in Python from the Step 13 assessments; the LLM never produces a number.

Scoring logic:
- supported = 1
- not_supported = 0
- Required requirements = 80% of the total score
- Preferred requirements = 20% of the total score
So:
Match Score = (Required support rate × 80) + (Preferred support rate × 20)

In [15]:
required = [
    a for a in candidate_requirement_assessments
    if a.requirement_type == "required"
]

preferred = [
    a for a in candidate_requirement_assessments
    if a.requirement_type == "preferred"
]

required_rate = (
    sum(a.status == "supported" for a in required) / len(required)
    if required else 0
)

preferred_rate = (
    sum(a.status == "supported" for a in preferred) / len(preferred)
    if preferred else 0
)

if required and preferred:
    match_score = (required_rate * 80) + (preferred_rate * 20)
elif required:
    match_score = required_rate * 100
elif preferred:
    match_score = preferred_rate * 100
else:
    match_score = 0

match_score = round(match_score, 2)

print(f"Required requirements supported: {sum(a.status == 'supported' for a in required)}/{len(required)}")
print(f"Preferred requirements supported: {sum(a.status == 'supported' for a in preferred)}/{len(preferred)}")
print(f"Match Score: {match_score}/100")

Required requirements supported: 8/13
Preferred requirements supported: 0/3
Match Score: 49.23/100


#### Step 14 — Insights summary
- The match score for Aarav Mehta is 49.23/100.
- The resume supports 8 of 13 required requirements and 0 of 3 preferred requirements.
- Required criteria carry 80% of the total score and preferred criteria carry 20%, so the score here comes entirely from the required requirements that are supported.
- The score is calculated in Python from the Step 13 assessments; the LLM never produces a number.
- Given the same Step 13 assessments, the score is always the same, which makes it transparent and easy to check by hand.
- The same scoring rule is applied to every candidate, so scores can be compared fairly.

**Interpretation:** Aarav shows a moderate match to the JD. Five required requirements and all three preferred requirements are not demonstrated by the resume.

----

### Step 15 — Derive Matching & Missing skills/requirements
**Objective:**

Convert the grounded Step 13 assessments into clear matching and missing requirement lists for the candidate, while retaining whether each item is required or preferred and its category.

Logic:
This step should be completely deterministic — no LLM call is needed.
- supported → matching requirement
- not_supported → missing requirement
- Carry forward:
  - requirement text
  - required / preferred
  - category
- Do not reinterpret, rename, or infer anything.
- “Missing” means not demonstrated by the resume, not that the candidate definitely lacks the capability.

In [16]:
matching_requirements = [
    {
        "requirement": a.requirement,
        "requirement_type": a.requirement_type,
        "category": a.category
    }
    for a in candidate_requirement_assessments
    if a.status == "supported"
]

missing_requirements = [
    {
        "requirement": a.requirement,
        "requirement_type": a.requirement_type,
        "category": a.category
    }
    for a in candidate_requirement_assessments
    if a.status == "not_supported"
]

print("Matching requirements:\n")
for item in matching_requirements:
    print(
        f"- {item['requirement_type'].upper()} | "
        f"{item['category']} | "
        f"{item['requirement']}"
    )

print("\nMissing / unsupported requirements:\n")
for item in missing_requirements:
    print(
        f"- {item['requirement_type'].upper()} | "
        f"{item['category']} | "
        f"{item['requirement']}"
    )

print(f"\nMatching requirements: {len(matching_requirements)}")
print(f"Missing requirements: {len(missing_requirements)}")

Matching requirements:

- REQUIRED | skill | Proficiency with data mining
- REQUIRED | skill | Proficiency with statistical analysis
- REQUIRED | tool | Experience with Excel
- REQUIRED | tool | Experience with PowerPoint
- REQUIRED | tool | Experience with Tableau
- REQUIRED | tool | Experience with SQL
- REQUIRED | tool | Experience with programming languages (ex: Java/Python, SAS)
- REQUIRED | skill | Ability to work effectively on several concurrent projects

Missing / unsupported requirements:

- REQUIRED | experience | Seven or more years of experience in data science
- REQUIRED | skill | Proficiency with mathematics
- REQUIRED | skill | Advanced experience in pattern recognition
- REQUIRED | skill | Advanced experience in predictive modeling
- REQUIRED | skill | Ability to work effectively in a dynamic, research-oriented group
- PREFERRED | education | Bachelor’s degree (or equivalent) in statistics, applied mathematics, or related discipline
- PREFERRED | experience | Two or mo

#### Short insights summary
Aarav matches 8 of 16 total requirements. His strengths are concentrated in core data science tools and skills such as data mining, statistical analysis, Excel, PowerPoint, Tableau, SQL, programming languages, and concurrent-project work. The main unsupported areas are the 7+ years experience threshold, mathematics proficiency, advanced pattern recognition/predictive modeling, research-oriented group experience, and all 3 preferred qualifications.

-----

### Step 16 — Create Grounded Final Evaluation Prompt

Objective:
Decide the hiring recommendation in Python, and create a grounded LLM prompt that generates only the two qualitative fields:
- Candidate Summary
- Recommendation Justification

The deterministic match score from Step 14 must remain unchanged and must not be recalculated by the LLM.

Hiring recommendation rule (decided in Python from the match score):
- 80 and above → Recommend for hire
- 60 to 79.99 → Consider for interview
- below 60 → Not recommended for hire

Logic:
The prompt will receive:
- candidate name
- deterministic match_score
- the fixed hiring recommendation
- grounded requirement assessments from Step 13, including each reason and its supporting evidence

Rules:
- Unsupported means “not demonstrated by the resume,” not that the candidate definitely lacks the capability.
- The LLM cannot change the score or the recommendation, or invent new candidate facts.
- The justification explains the fixed recommendation using the supported and unsupported requirements.
- Strengths and weaknesses are not written by the LLM. Step 17 builds them in Python from the Step 13 assessments.

In [17]:
class QualitativeEvaluation(BaseModel):
    candidate_summary: str = Field(
        description="Concise grounded summary of the candidate's fit for the JD"
    )
    recommendation_justification: str = Field(
        description="Three to five sentences justifying the fixed hiring recommendation"
    )


qualitative_parser = PydanticOutputParser(
    pydantic_object=QualitativeEvaluation
)


def recommendation_from_score(score):
    if score >= 80:
        return "Recommend for hire"
    if score >= 60:
        return "Consider for interview"
    return "Not recommended for hire"


qualitative_prompt = ChatPromptTemplate.from_template("""
Create the final qualitative evaluation for one candidate.

Use ONLY:
- the grounded requirement assessments,
- the deterministic match score,
- the fixed hiring recommendation below.

Do not recalculate the score.
Do not change the recommendation.
Do not invent candidate facts.
Unsupported means "not demonstrated in the resume".
Where an assessment reason states a specific fact, such as a number of
years, include that fact.

Generate only:
- candidate_summary
- recommendation_justification

candidate_summary:
- concise but informative;
- mention the candidate's strongest demonstrated areas;
- mention the most important unsupported requirements where relevant.

recommendation_justification:
- write 3 to 5 clear recruiter-ready sentences;
- explain why the candidate is or is not a fit;
- name the most important demonstrated strengths;
- name the most important unsupported REQUIRED requirements;
- mention preferred gaps separately where useful;
- do not use vague blanket statements without naming the actual reasons.

CANDIDATE:
{candidate_name}

DETERMINISTIC MATCH SCORE:
{match_score}/100

FIXED HIRING RECOMMENDATION:
{hiring_recommendation}

GROUNDED REQUIREMENT ASSESSMENTS:
{assessment_context}

{format_instructions}
""")


qualitative_chain = (
    qualitative_prompt
    | llm
    | qualitative_parser
)

print("Grounded final evaluation prompt created successfully.")

Grounded final evaluation prompt created successfully.


### Step 17 — Assemble final candidate evaluation

Objective
Combine:
- deterministic outputs from Steps 14–15, and
- grounded qualitative outputs from Step 16
into the final evaluation schema defined earlier in Step 8.

Logic
Step 17 should:
1. Build a compact assessment_context from the grounded Step 13 assessments.
2. Invoke the Step 16 qualitative chain once.
3. Assemble the final result using:
   - candidate name
   - deterministic match_score
   - matching requirements
   - missing requirements
   - strengths: each supported requirement with its resume evidence, built in Python
   - weaknesses: each unsupported required requirement with its reason, built in Python
   - hiring recommendation from the Step 16 score rule
   - candidate summary and recommendation justification written by the LLM
4. The LLM must not overwrite deterministic fields.

In [18]:
assessment_context = "\n\n".join(
    (
        f"REQUIREMENT: {a.requirement}\n"
        f"TYPE: {a.requirement_type.upper()}\n"
        f"STATUS: {a.status.upper()}\n"
        f"REASON: {a.assessment_reason}\n"
        f"EVIDENCE: "
        f"{'; '.join(a.supporting_evidence) if a.supporting_evidence else 'None'}"
    )
    for a in candidate_requirement_assessments
)

hiring_recommendation = recommendation_from_score(match_score)

qualitative_evaluation = qualitative_chain.invoke({
    "candidate_name": candidate_name,
    "match_score": match_score,
    "hiring_recommendation": hiring_recommendation,
    "assessment_context": assessment_context,
    "format_instructions": qualitative_parser.get_format_instructions()
})

# Strengths and weaknesses are built in Python from the Step 13 assessments
strengths = [
    (
        f"{a.requirement}: {a.supporting_evidence[0]}"
        if a.supporting_evidence
        else a.requirement
    )
    for a in candidate_requirement_assessments
    if a.status == "supported"
]

weaknesses = [
    f"{a.requirement}: {a.assessment_reason}"
    for a in candidate_requirement_assessments
    if a.status == "not_supported"
    and a.requirement_type == "required"
]

final_candidate_evaluation = CandidateEvaluation(
    candidate_name=candidate_name,
    match_score=match_score,
    matching_skills=[
        item["requirement"]
        for item in matching_requirements
    ],
    missing_skills=[
        item["requirement"]
        for item in missing_requirements
    ],
    candidate_summary=qualitative_evaluation.candidate_summary,
    strengths=strengths,
    weaknesses=weaknesses,
    hiring_recommendation=hiring_recommendation,
    recommendation_justification=qualitative_evaluation.recommendation_justification
)

print(final_candidate_evaluation.model_dump_json(indent=2))

{
  "candidate_name": "Aarav Mehta",
  "match_score": 49.23,
  "matching_skills": [
    "Proficiency with data mining",
    "Proficiency with statistical analysis",
    "Experience with Excel",
    "Experience with PowerPoint",
    "Experience with Tableau",
    "Experience with SQL",
    "Experience with programming languages (ex: Java/Python, SAS)",
    "Ability to work effectively on several concurrent projects"
  ],
  "missing_skills": [
    "Seven or more years of experience in data science",
    "Proficiency with mathematics",
    "Advanced experience in pattern recognition",
    "Advanced experience in predictive modeling",
    "Ability to work effectively in a dynamic, research-oriented group",
    "Bachelor’s degree (or equivalent) in statistics, applied mathematics, or related discipline",
    "Two or more years of project management experience",
    "Professional certification"
  ],
  "candidate_summary": "Aarav Mehta has 6 years of experience in data science, demonstrating 

### Step 18 — Evaluate all candidates & compare

Objective
Run the same locked evaluation workflow independently for every candidate resume, then compare candidates using their deterministic match scores.

Logic
For each candidate:
1. Use only that candidate’s retriever — preserving resume isolation.
2. Apply the same Step 13 requirement-assessment logic.
3. Calculate the same Step 14 80% required / 20% preferred deterministic score.
4. Derive matching and missing requirements as in Step 15.
5. Generate the grounded qualitative evaluation using Steps 16–17.
6. Store the final structured evaluation.
7. Rank candidates by deterministic match score.
No candidate's evidence is allowed to influence another candidate.

Output
- a candidate ranking table;
- a requirement-by-candidate comparison table;
- an audit trail giving the reason for every requirement not demonstrated.

In [19]:
# Step 18 — Evaluate all candidates and compare

import pandas as pd


def calculate_match_score(assessments):
    required = [a for a in assessments if a.requirement_type == "required"]
    preferred = [a for a in assessments if a.requirement_type == "preferred"]

    required_rate = (
        sum(a.status == "supported" for a in required) / len(required)
        if required else 0
    )
    preferred_rate = (
        sum(a.status == "supported" for a in preferred) / len(preferred)
        if preferred else 0
    )

    if required and preferred:
        score = required_rate * 80 + preferred_rate * 20
    elif required:
        score = required_rate * 100
    elif preferred:
        score = preferred_rate * 100
    else:
        score = 0

    return round(score, 2)


def evaluate_candidate(candidate_id):
    candidate_name = (
        candidate_chunks[candidate_id][0].metadata["candidate_name"]
    )

    assessments = assess_candidate(candidate_id)
    score = calculate_match_score(assessments)
    hiring_recommendation = recommendation_from_score(score)

    assessment_context = "\n\n".join(
        (
            f"REQUIREMENT: {a.requirement}\n"
            f"TYPE: {a.requirement_type.upper()}\n"
            f"STATUS: {a.status.upper()}\n"
            f"REASON: {a.assessment_reason}\n"
            f"EVIDENCE: "
            f"{'; '.join(a.supporting_evidence) if a.supporting_evidence else 'None'}"
        )
        for a in assessments
    )

    qualitative = qualitative_chain.invoke({
        "candidate_name": candidate_name,
        "match_score": score,
        "hiring_recommendation": hiring_recommendation,
        "assessment_context": assessment_context,
        "format_instructions": qualitative_parser.get_format_instructions()
    })

    evaluation = CandidateEvaluation(
        candidate_name=candidate_name,
        match_score=score,
        matching_skills=[
            a.requirement for a in assessments
            if a.status == "supported"
        ],
        missing_skills=[
            a.requirement for a in assessments
            if a.status == "not_supported"
        ],
        candidate_summary=qualitative.candidate_summary,
        strengths=[
            (
                f"{a.requirement}: {a.supporting_evidence[0]}"
                if a.supporting_evidence
                else a.requirement
            )
            for a in assessments
            if a.status == "supported"
        ],
        weaknesses=[
            f"{a.requirement}: {a.assessment_reason}"
            for a in assessments
            if a.status == "not_supported"
            and a.requirement_type == "required"
        ],
        hiring_recommendation=hiring_recommendation,
        recommendation_justification=qualitative.recommendation_justification
    )

    return evaluation, assessments


all_candidate_evaluations = {}
all_candidate_assessments = {}

for candidate_id in candidate_retrievers:
    evaluation, assessments = evaluate_candidate(candidate_id)
    all_candidate_evaluations[candidate_id] = evaluation
    all_candidate_assessments[candidate_id] = assessments


# Ranking table
ranking_df = pd.DataFrame([
    {
        "Candidate": e.candidate_name,
        "Match Score": e.match_score,
        "Supported": len(e.matching_skills),
        "Not demonstrated": len(e.missing_skills),
        "Recommendation": e.hiring_recommendation
    }
    for e in all_candidate_evaluations.values()
]).sort_values("Match Score", ascending=False)

ranking_df.insert(0, "Rank", range(1, len(ranking_df) + 1))

print("Candidate ranking:\n")
print(ranking_df.to_string(index=False))


# Requirement-by-candidate comparison table
comparison_df = pd.DataFrame({
    "Requirement": [
        r.requirement for r in jd_requirements.requirements
    ],
    "Type": [
        r.requirement_type for r in jd_requirements.requirements
    ],
    **{
        all_candidate_evaluations[cid].candidate_name: [
            "Yes" if a.status == "supported" else "No"
            for a in all_candidate_assessments[cid]
        ]
        for cid in all_candidate_assessments
    }
})

print("\nRequirement comparison:\n")
print(comparison_df.to_string(index=False))


# Audit trail: why each requirement was not demonstrated
print("\nReasons for each requirement not demonstrated:\n")

for cid, candidate_assessments in all_candidate_assessments.items():
    print(all_candidate_evaluations[cid].candidate_name)

    for a in candidate_assessments:
        if a.status == "not_supported":
            print(f"  NO | {a.requirement}")
            print(f"       Reason: {a.assessment_reason}")

    print()

Candidate ranking:

 Rank   Candidate  Match Score  Supported  Not demonstrated           Recommendation
    1 Rhea Sharma        68.72         11                 5   Consider for interview
    2 Aarav Mehta        49.23          8                 8 Not recommended for hire
    3 Arjun Mehta        24.62          4                12 Not recommended for hire

Requirement comparison:

                                                                                Requirement      Type Aarav Mehta Rhea Sharma Arjun Mehta
                                          Seven or more years of experience in data science  required          No          No          No
                                                               Proficiency with data mining  required         Yes         Yes          No
                                                               Proficiency with mathematics  required          No          No          No
                                                      Proficie

### Step 19 — Required validation tests

**Objective**
Explicitly demonstrate that the completed pipeline satisfies the assignment's required evaluation scenarios, using the results already generated in Step 18.

This step tests:
1. Resume A individual evaluation
2. Resume A vs Resume B comparison
3. Resume C missing skills
4. Best-candidate selection
5. Hiring recommendation with justification

Each scenario prints its result and confirms it against facts stated in the resumes.


In [20]:
# Step 19 — Required validation tests from assignment asks 

ALLOWED_RECOMMENDATIONS = {
    "Recommend for hire",
    "Consider for interview",
    "Not recommended for hire"
}


def status_of(candidate_id, keyword):
    """Return the status of the one requirement containing the keyword."""
    matches = [
        a for a in all_candidate_assessments[candidate_id]
        if keyword.lower() in a.requirement.lower()
    ]
    assert len(matches) == 1, (
        f"Expected one requirement containing '{keyword}', "
        f"found {len(matches)}."
    )
    return matches[0].status


resume_a = all_candidate_evaluations["A"]
resume_b = all_candidate_evaluations["B"]
resume_c = all_candidate_evaluations["C"]


# TEST 1 — Evaluate Resume A for the Data Scientist role
print("TEST 1 — Resume A individual evaluation")
print(f"Candidate: {resume_a.candidate_name}")
print(f"Match Score: {resume_a.match_score}/100")
print(f"Recommendation: {resume_a.hiring_recommendation}")
print(f"Summary: {resume_a.candidate_summary}")

assert status_of("A", "Seven or more years") == "not_supported", (
    "Resume A states 6 years, so the 7-year requirement must not be supported."
)
assert status_of("A", "Tableau") == "supported", (
    "Resume A describes building Tableau dashboards."
)
assert status_of("A", "certification") == "not_supported", (
    "Resume A lists no certification."
)
print("PASSED\n")


# TEST 2 — Compare Resume A and Resume B for the same JD
print("TEST 2 — Resume A vs Resume B comparison")
print(f"{resume_a.candidate_name}: {resume_a.match_score}/100")
print(f"{resume_b.candidate_name}: {resume_b.match_score}/100")
print("Requirements where they differ:")

for a, b in zip(
    all_candidate_assessments["A"],
    all_candidate_assessments["B"]
):
    if a.status != b.status:
        print(
            f"- {a.requirement} | "
            f"{resume_a.candidate_name}: {a.status} | "
            f"{resume_b.candidate_name}: {b.status}"
        )

assert resume_b.match_score > resume_a.match_score, (
    "Resume B should score higher than Resume A."
)
assert status_of("B", "certification") == "supported", (
    "Resume B lists the Certified Analytics Professional certification."
)
assert status_of("B", "research-oriented") == "supported", (
    "Resume B describes working in a research-oriented analytics group."
)
print("PASSED\n")


# TEST 3 — Identify missing skills in Resume C
print("TEST 3 — Resume C missing skills / requirements")
print(f"Candidate: {resume_c.candidate_name}")
for item in resume_c.missing_skills:
    print(f"- {item}")

for keyword in ["Tableau", "PowerPoint", "data mining"]:
    assert status_of("C", keyword) == "not_supported", (
        f"Resume C does not mention {keyword}."
    )

assert status_of("C", "statistical analysis") == "not_supported", (
    "Resume C only expresses a wish for exposure to statistical analysis, "
    "which is not evidence."
)
assert status_of("C", "SQL") == "supported", (
    "Resume C describes writing SQL queries."
)
print("PASSED\n")


# TEST 4 — Recommend the best candidate among multiple resumes
best_candidate = max(
    all_candidate_evaluations.values(),
    key=lambda e: e.match_score
)

print("TEST 4 — Best candidate selection")
print(f"Best Candidate: {best_candidate.candidate_name}")
print(f"Match Score: {best_candidate.match_score}/100")

assert best_candidate.candidate_name == "Rhea Sharma", (
    "Rhea Sharma should be the highest-ranked candidate."
)
assert resume_a.match_score > resume_c.match_score, (
    "Resume A should rank above Resume C."
)
print("PASSED\n")


# TEST 5 — Hiring recommendation with justification
print("TEST 5 — Hiring recommendation with justification")

for evaluation in all_candidate_evaluations.values():
    print(f"Candidate: {evaluation.candidate_name}")
    print(f"Recommendation: {evaluation.hiring_recommendation}")
    print(f"Justification: {evaluation.recommendation_justification}\n")

    assert evaluation.hiring_recommendation in ALLOWED_RECOMMENDATIONS, (
        "The recommendation must be one of the three fixed labels."
    )
    assert evaluation.hiring_recommendation == recommendation_from_score(
        evaluation.match_score
    ), "The recommendation must follow from the match score."
    assert evaluation.recommendation_justification.strip(), (
        "A justification is required."
    )

print("PASSED\n")

print("All 5 Step 19 validation tests passed.")

TEST 1 — Resume A individual evaluation
Candidate: Aarav Mehta
Match Score: 49.23/100
Recommendation: Not recommended for hire
Summary: Aarav Mehta has 6 years of experience in data science, demonstrating strong proficiency in data mining, statistical analysis, and tools such as Python, SQL, Tableau, and Excel. However, he does not meet the required seven years of experience in data science and lacks demonstrated proficiency in mathematics, advanced experience in pattern recognition, and predictive modeling.
PASSED

TEST 2 — Resume A vs Resume B comparison
Aarav Mehta: 49.23/100
Rhea Sharma: 68.72/100
Requirements where they differ:
- Ability to work effectively in a dynamic, research-oriented group | Aarav Mehta: not_supported | Rhea Sharma: supported
- Bachelor’s degree (or equivalent) in statistics, applied mathematics, or related discipline | Aarav Mehta: not_supported | Rhea Sharma: supported
- Professional certification | Aarav Mehta: not_supported | Rhea Sharma: supported
PASSED